In [1]:
import json
import math
import statistics
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime
from typing import Any, List, Dict, Tuple, Optional

In [2]:
def to_float(value: Any) -> Optional[float]:
    if value is None or value == "":
        return None
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        clean = value.strip().replace("R$", "").replace(".", "").replace(",", ".")
        try:
            return float(clean)
        except ValueError:
            return None

teste = to_float("R$ 1.234,56")
print(teste)

1234.56


In [3]:
def to_bool(value: Any) -> Optional[bool]:
    if value is None or value == "":
        return None
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)):
        return bool(value)
    if isinstance(value, str):
        text = value.strip().lower()

        if text in ("true", "1", "sim", "s", "yes", "y"):
            return True

        if text in ("false", "0", "nao", "não", "n", "no"):
            return False

    return None

teste = to_bool("Sim")
print(teste)

True


In [4]:
def parse_date(value: Any) -> Optional[str]:
    if value is None or value == "":
        return None

    if isinstance(value, str):
        for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%Y/%m/%d", "%d-%m-%Y"):
            try:
                return datetime.strptime(value, fmt).strftime("%Y-%m-%d")
            except ValueError:
                continue
        return value

    return None

teste = parse_date("31/12/2020")
print(teste)    

2020-12-31


In [5]:
def infer_field_type(values: List[Any]) -> str:
    not_null = [v for v in values if v is not None and v != ""]

    if not not_null:
        return "indefinido"

    numeric = sum(
        1 for v in not_null
        if isinstance(v, (int, float)) or to_float(v) is not None
    )

    if numeric == len(not_null):
        unique = len(set(float(to_float(v)) for v in not_null if to_float(v) is not None))

        if unique <= 12:
            return "quantitativa discreta"

        return "quantitativa continua"

    unique = len(set(str(v) for v in not_null))

    if unique <= 12:
        return "qualitativa nominal"

    return "qualitativa_ordinal_ou_textual"

print(infer_field_type(["1", "2", "3", "4", "5"]))

quantitativa discreta


# Testes para to_float

In [7]:

print(to_float("123"))
print(to_float("-45,6"))
print(to_float("  7.89 "))
print(to_float("R$ 1.234,56"))
print(to_float(42))
print(to_float(0))
print(to_float(3.14))
print(to_float(0.0))
print(to_float(""))
print(to_float("abc"))

123.0
-45.6
789.0
1234.56
42.0
0.0
3.14
0.0
None
None


# Testes para to_bool

In [8]:
print(to_bool("true"))
print(to_bool("False"))
print(to_bool("Sim"))
print(to_bool("n"))
print(to_bool(1))
print(to_bool(0))
print(to_bool(0.0))
print(to_bool(True))
print(to_bool(False))
print(to_bool("maybe"))

True
False
True
False
True
False
False
True
False
None


# Testes para parse_date

In [9]:
print(parse_date("2020-01-02"))
print(parse_date("02/01/2020"))
print(parse_date("2020/01/02"))
print(parse_date("02-01-2020"))
print(parse_date(""))
print(parse_date(None))
print(parse_date("31/02/2020"))
print(parse_date("abc"))
print(parse_date("2021-13-01"))
print(parse_date(datetime(2020, 12, 31)))

2020-01-02
2020-01-02
2020-01-02
2020-01-02
None
None
31/02/2020
abc
2021-13-01
None


# Testes para infer_field_type

In [10]:
print(infer_field_type(["1", "2", "3", "4", "5"]))
print(infer_field_type(["1.1", "2.2", "3.3", "4.4", "5.5", "6.6", "7.7", "8.8", "9.9", "10.1", "11.2", "12.3", "13.4"]))
print(infer_field_type(["a", "b", "a", "b"]))
print(infer_field_type(["alto", "médio", "baixo", "médio", "alto", "baixo"]))
print(infer_field_type(["vermelho", "verde", "azul", "amarelo", "roxo", "laranja", "cinza", "preto", "branco", "marrom", "rosa", "bege", "turquesa"]))
print(infer_field_type([None, "", None]))
print(infer_field_type(["1", "2", "2", "2", "3", "3", "3", "4", "5", "5", "5", "6"]))
print(infer_field_type([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]))
print(infer_field_type(["True", "False", "True"]))
print(infer_field_type(["10", "20", "30", "40", "50", "60", "70", "80", "90", "100", "110", "120"]))

quantitativa discreta
quantitativa continua
qualitativa nominal
qualitativa nominal
qualitativa_ordinal_ou_textual
indefinido
quantitativa discreta
quantitativa continua
qualitativa nominal
quantitativa discreta
